# Day 8 content

In [3]:
import json
from traceback import print_tb

raw = '{"category": "TRACK_ORDER", "confidence": "high"}'
data = json.loads(raw)

if 'category' not in data:
    raise ValueError('missing category')
if data['category'] not in ('TRACK_ORDER', 'CANCEL', 'REFUND'):
    raise ValueError(f'Invalid catefory: {data['category']}')
if 'confidence' not in data:
    raise ValueError('missing confidence')
print('Manual validation passed:', data)

Manual validation passed: {'category': 'TRACK_ORDER', 'confidence': 'high'}


In [9]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Literal

class CustomerRecord(BaseModel):
    customer_id : int
    first_name  : str
    last_name   : str
    email       : str
    city        : str
    state       : str

customer = CustomerRecord(
    customer_id = 1001,
    first_name  = 'Yogi',
    last_name   = 'Reddy',
    email       = 'yogireddy@gmail.com',
    city        = 'Bengaluru',
    state       = 'KA'
)

print(repr(customer))
print()
print(str(customer))
print()
print('customer_id_type :', type(customer.customer_id).__name__)
print('first_name       :', customer.first_name)

CustomerRecord(customer_id=1001, first_name='Yogi', last_name='Reddy', email='yogireddy@gmail.com', city='Bengaluru', state='KA')

customer_id=1001 first_name='Yogi' last_name='Reddy' email='yogireddy@gmail.com' city='Bengaluru' state='KA'

customer_id_type : int
first_name       : Yogi


In [10]:
from_csv = CustomerRecord(
    customer_id = '1001',
    first_name  = 'Yogi',
    last_name   = 'Reddy',
    email       = 'yogireddy@gmail.com',
    city        = 'Bengaluru',
    state       = 'KA'
)

print('customer_id:', from_csv.customer_id, '- type:', type(customer.customer_id))

customer_id: 1001 - type: <class 'int'>


In [12]:
class CustomerRecord(BaseModel):
    customer_id : int = Field(gt=0,description='Must be positive integer')
    first_name  : str = Field(min_length=1, max_length=50)
    last_name   : str = Field(min_length=1, max_length=50)
    email       : str = Field(min_length=5)
    city        : str
    state       : str = Field(min_length=2, max_length=2, description='Two-letter state code')
    country     : str = Field(default='India')

    @property
    def full_name(self):
        return f'{self.first_name} {self.last_name}'

c = CustomerRecord(
    customer_id=1001, first_name= 'Yogi', last_name= 'Reddy',email='yogireddy@gmail.com', city='Bengaluru', state='KA'
)
print(c.full_name, '|', c.country)

Yogi Reddy | India


In [14]:
try:
    bad = CustomerRecord(
    customer_id = -5,
    first_name  = '',
    last_name   = 'Reddy',
    email       = 'y@gmail.com',
    city        = 'Bengaluru',
    state       = 'KA'
    )
except ValidationError as e:
    print('ValidationError - field-level details:')
    for err in e.errors():
        print(f' field: {err['loc']} -> {err['msg']}')

ValidationError - field-level details:
 field: ('customer_id',) -> Input should be greater than 0
 field: ('first_name',) -> String should have at least 1 character


In [15]:
from typing import Optional, Literal
class ProductRecord(BaseModel):
    product_id  : int = Field(gt=0)
    product_name  : str   = Field(min_length=1)
    category      : str
    price         : float = Field(gt=0)
    stock_quantity: int   = Field(ge=0)
    description   : Optional[str] = None

    @property
    def is_in_stock(self):
        return self.stock_quantity > 0

    @property
    def price_str(self):
        return f'${self.price:.2f}'

    def __repr__(self):
        return f'ProductRecord(id={self.product_id}, name={self.product_name!r}, price={self.price_str})'

p1 = ProductRecord(product_id=101, product_name='Classic Monitor',
                   category='Electronics', price=205.21, stock_quantity=238)
print(p1)
print('in stock:', p1.is_in_stock)
print('description:', p1.description)

product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238 description=None
in stock: True
description: None


In [17]:
class OrderRecord(BaseModel):
    order_id       : int
    customer_id    : int
    total_amount   : float = Field(ge=0)
    payment_method : str
    # Literal enforces that status must be one of these exact strings
    order_status   : Literal['Delivered', 'In Transit', 'Processing','Pending', 'Cancelled', 'Refunded']

o = OrderRecord(order_id=3042, customer_id=1001,
                total_amount=205.21, payment_method='Credit Card',
                order_status='In Transit')
print(o)

try:
    bad = OrderRecord(order_id=3043, customer_id=1001,
                      total_amount=50.0, payment_method='Cash',
                      order_status='Shipped')
except ValidationError as e:
    print()
    print("Invalid status caught:")
    for err in e.errors():
        print(f"  {err['loc']}  ->  {err['msg']}")

order_id=3042 customer_id=1001 total_amount=205.21 payment_method='Credit Card' order_status='In Transit'

Invalid status caught:
  ('order_status',)  ->  Input should be 'Delivered', 'In Transit', 'Processing', 'Pending', 'Cancelled' or 'Refunded'


In [21]:
row = {
    'product_id': '101',       # str — Pydantic converts to int
    'product_name': 'Classic Monitor',
    'category': 'Electronics',
    'price': '205.21',         # str — Pydantic converts to float
    'stock_quantity': '238',
}

product = ProductRecord(**row)
print(product)
product = ProductRecord.model_validate(row)
print(product)
print('price type   :', type(product.price).__name__)
print('stock type   :', type(product.stock_quantity).__name__)

product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238 description=None
product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238 description=None
price type   : float
stock type   : int


In [22]:
json_str = '{"product_id": 102, "product_name": "Yoga Mat", "category": "Sports", "price": 45.0, "stock_quantity": 150}'

product2 = ProductRecord.model_validate_json(json_str)
print(product2)

product_id=102 product_name='Yoga Mat' category='Sports' price=45.0 stock_quantity=150 description=None


In [25]:
product = ProductRecord(product_id=101, product_name='Classic Monitor',
                        category='Electronics', price=205.21, stock_quantity=238)

d = product.model_dump()
print(type(d).__name__, d)

s = product.model_dump_json()
print(type(s).__name__, s)

d['price'] = round(d['price'] * 0.9, 2)
print('After discount:', d['price'])

dict {'product_id': 101, 'product_name': 'Classic Monitor', 'category': 'Electronics', 'price': 205.21, 'stock_quantity': 238, 'description': None}
str {"product_id":101,"product_name":"Classic Monitor","category":"Electronics","price":205.21,"stock_quantity":238,"description":null}
After discount: 184.69


In [26]:
class TriageOutput:

    model_config = {'str_strip_whitespace': True}

    def __init__(self, category, confidence, reason):
        if category not in ['TRACK_ORDER', 'CANCEL', 'REFUND', 'GENERAL']:
            raise ValidationError("Category must be one of 'TRACK_ORDER', 'CANCEL', 'REFUND', 'GENERAL'")

        if confidence not in ['high', 'medium', 'low']:
            raise ValidationError("Confidence must be one of 'high', 'medium', 'low'")

        if len(reason) > 5:
            raise ValidationError("Reason must be less than 5 characters")

        self.category = category
        self.confidence = confidence
        self.reason = reason

In [27]:
class TriageOutput(BaseModel):
    category   : Literal['TRACK_ORDER', 'CANCEL', 'REFUND', 'GENERAL']
    confidence : Literal['high', 'medium', 'low']
    reason     : str = Field(min_length=5)

    model_config = {'str_strip_whitespace': True}

def parse_llm_response(raw_respose):

    cleaned = raw_respose.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1].rsplit('```',1)[0].strip()

    return TriageOutput.model_validate_json(cleaned)

# Test 1 — plain JSON
r1 = parse_llm_response('{"category": "TRACK_ORDER", "confidence": "high", "reason": "Customer asked about order location"}')
print('Test 1:', r1.category, r1.confidence)

Test 1: TRACK_ORDER high


In [28]:
# Test 2 — LLM wrapped in markdown fences (very common)
fenced = '```json\n{"category": "CANCEL", "confidence": "medium", "reason": "Customer wants to cancel"}\n```'
r2 = parse_llm_response(fenced)
print('Test 2:', r2.category, r2.confidence)

Test 2: CANCEL medium


In [29]:
# Test 3 — LLM returned invalid category
try:
    r3 = parse_llm_response('{"category": "UNKNOWN", "confidence": "high", "reason": "Not sure"}')
except ValidationError as e:
    print('Test 3 — caught invalid category:')
    for err in e.errors():
        print(f"  field {err['loc']} → {err['msg']}")

Test 3 — caught invalid category:
  field ('category',) → Input should be 'TRACK_ORDER', 'CANCEL', 'REFUND' or 'GENERAL'


In [30]:
# Test 4 — LLM missed a required field
try:
    r4 = parse_llm_response('{"category": "CANCEL", "confidence": "high"}')
    # 'reason' field missing
except ValidationError as e:
    print('Test 4 — missing field:')
    for err in e.errors():
        print(f"  field {err['loc']} → {err['msg']}")

Test 4 — missing field:
  field ('reason',) → Field required


In [36]:
from typing import List
from pydantic import BaseModel


# Nested model representing employee address details
class Address(BaseModel):
    street: str
    city: str
    state: str


# Nested model representing project/work experience details
class Project(BaseModel):
    project_name: str
    experience: int
    company: str
    is_current_org: bool


# Main model representing an employee record
class EmployeeRecord(BaseModel):
    name: str
    age: int
    salary: int
    department: str

    # Nested Address object
    address: Address

    # List of Project objects
    projects: List[Project]


# Sample employee data received from a file, API, database, etc.
record = {
    "name": "Prudhvi",
    "age": 30,
    "salary": 20000,
    "department": "IT",

    # Address information
    "address": {
        "street": "DNO: XYZ",
        "city": "Rajahmundry",
        "state": "AP",
    },

    # List of projects worked on by the employee
    "projects": [
        {
            "project_name": "XYZ",
            "experience": 2,
            "company": "ABC",
            "is_current_org": True
        },
        {
            "project_name": "AMZ",
            "experience": 4,
            "company": "BNC",
            "is_current_org": False
        }
    ]
}

employee = EmployeeRecord.model_validate(record)
print(employee)

print(employee.name)
print(employee.address.city)
print(employee.projects[0].project_name)

name='Prudhvi' age=30 salary=20000 department='IT' address=Address(street='DNO: XYZ', city='Rajahmundry', state='AP') projects=[Project(project_name='XYZ', experience=2, company='ABC', is_current_org=True), Project(project_name='AMZ', experience=4, company='BNC', is_current_org=False)]
Prudhvi
Rajahmundry
XYZ
